In [25]:
from google.colab import files

In [26]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
train_df = pd.read_csv('/content/drive/MyDrive/train.csv')
test_df = pd.read_csv('/content/drive/MyDrive/test.csv')

In [28]:
import os
print(os.listdir('/content'))

['.config', 'final_model', 'drive', 'model_output', 'sample_data']


In [ ]:
# pandas is already imported in the earlier cell

In [30]:
print("Training samples:", len(train_df))
print("Test samples:", len(test_df))

Training samples: 2735
Test samples: 684


In [31]:
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq
)
from datasets import Dataset

MODEL_NAME = "facebook/bart-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_ds = Dataset.from_pandas(train_df)
test_ds = Dataset.from_pandas(test_df)

MAX_INPUT = 512
MAX_TARGET = 128

def preprocess(batch):
    inputs = tokenizer(
        batch["findings"],
        max_length=MAX_INPUT,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        batch["impression"],
        max_length=MAX_TARGET,
        truncation=True,
        padding="max_length"
    )["input_ids"]

    # IMPORTANT FIX: replace pad token id with -100
    labels = [
        [(token if token != tokenizer.pad_token_id else -100) for token in seq]
        for seq in labels
    ]

    inputs["labels"] = labels
    return inputs

train_ds = train_ds.map(preprocess, batched=True)
test_ds = test_ds.map(preprocess, batched=True)

# remove original columns (important for Trainer stability)
train_ds = train_ds.remove_columns(["findings", "impression"])
test_ds = test_ds.remove_columns(["findings", "impression"])

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

training_args = Seq2SeqTrainingArguments(
    output_dir="./model_output",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    warmup_steps=100,
    predict_with_generate=True,
    eval_strategy="epoch",   # FIXED
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=True,
    logging_steps=50,
    report_to="none"
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    data_collator=data_collator
)

trainer.train()

model.save_pretrained("./final_model")
tokenizer.save_pretrained("./final_model")

print("Training complete!")

Map:   0%|          | 0/2735 [00:00<?, ? examples/s]

Map:   0%|          | 0/684 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss
1,1.291541,1.151689
2,0.926769,1.047008
3,0.679072,1.000655


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training complete!


In [32]:
from google.colab import drive
drive.mount('/content/drive')

!cp -r final_model /content/drive/MyDrive/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
